# Baseline walkthrough
This notebook runs the three modes of the agent system on the bundled sample projects and shows what each agent produced. It needs no LLM: the default backend is the deterministic template. Test execution is switched on automatically when PySpark, pytest, coverage and Java are installed.

In [ ]:
import sys, json
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))

from ptagent.config import load_config
from ptagent.agents.orchestrator import Orchestrator

cfg = load_config()
cfg.output_dir = REPO / 'outputs' / 'notebook_run'
SAMPLES = REPO / 'data' / 'sample_projects'

## Mode A: no tests exist
The router sees no tests, so the test generation agent plans which functions are testable, infers input schemas from the code, retrieves guidance from the knowledge base and writes pytest files. Functions that need file paths or are entry points are skipped with a reason.

In [ ]:
report_a = Orchestrator(cfg).run(SAMPLES / 'orders_no_tests', mode='auto')
gen = report_a['generation']
print('mode:', report_a['mode_resolved'], '| tests generated:', gen['tests_generated'])
for item in gen['skipped']:
    print('skipped', item['function'], '->', item['reason'])

In [ ]:
test_file = cfg.output_dir / 'orders_no_tests' / 'generated_tests' / 'test_orders_etl_cleaning_generated.py'
print(test_file.read_text()[:1800])

## Mode B: tests exist
The obsolete-code agent finds unused functions and a stale test import, the coverage-gap agent lists functions that no test references, and tests are generated for those gaps only (dead code is excluded).

In [ ]:
report_b = Orchestrator(cfg).run(SAMPLES / 'orders_with_tests', mode='auto')
print('mode:', report_b['mode_resolved'])
print('dead functions:', [d['function'] for d in report_b['obsolete_code']['dead_functions']])
print('stale imports:', [(s['test_file'], s['missing_name']) for s in report_b['obsolete_code']['stale_test_imports']])
print('untested:', report_b['coverage_gaps']['untested_functions'])
print('coverage:', report_b['coverage'])

## Mode C: runtime upgrade
The runtime-impact agent applies the rules whose version window falls inside the upgrade and reports impacted notebooks and modules.

In [ ]:
report_c = Orchestrator(cfg).run(SAMPLES / 'legacy_runtime', mode='C', from_dbr='13.3', to_dbr='17.3')
impact = report_c['runtime_impact']
print('findings:', len(impact['findings']), '| impacted notebooks:', impact['impacted_notebooks'])
for f in impact['findings'][:6]:
    print(f['rule_id'], f['file'], f['location'], '-', f['title'])

## What the agents did
Every run writes `run_trace.json`, a step-by-step record of which agent did what.

In [ ]:
trace = json.loads((cfg.output_dir / 'legacy_runtime' / 'run_trace.json').read_text())
for event in trace:
    print(event['step'], event['agent'], '-', event['action'])

## Evaluation
Run `python run.py evaluate` from the repository root for precision/recall on the labelled samples (see `docs/evaluation.md`).